# Megatron-LM

A refresher on **Megatron-LM** — NVIDIA's reference framework for training transformer language models that are too big to fit on one GPU. Its lasting contribution is a set of *model-parallelism* recipes — **tensor**, **pipeline**, and **sequence** parallelism — that shard a single model across hundreds-to-thousands of GPUs while keeping the math identical to the single-GPU model.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**The problem:** a modern LLM doesn't fit on one accelerator. A 175B-parameter model needs ~350 GB just for fp16 weights, and 4–6× that once you add optimizer states (Adam keeps two moments) and activations. A single H100 has 80 GB. You *must* split the model itself across devices — and do it without changing what the model computes.

**Plain data parallelism isn't enough.** Data parallelism (DDP/ZeRO) replicates the model on every GPU and splits the *batch*. That scales throughput but every GPU still needs a full copy of the model. Once the model exceeds one GPU's memory, replication is impossible.

**Megatron-LM's answer is model parallelism** — partition the *weights and activations* of each layer across GPUs:

- **Tensor parallelism (TP)** — split the matrices *inside* each layer (attention heads, MLP columns) across GPUs in the same node. Communication every layer, so TP stays within the fast NVLink domain (typically TP ≤ 8).
- **Pipeline parallelism (PP)** — split the *stack of layers* into stages on different nodes; activations flow stage→stage like a factory line. Cheap communication (just layer boundaries) but introduces pipeline "bubbles."
- **Sequence parallelism (SP)** — shard the few remaining replicated ops (LayerNorm, dropout) along the sequence dimension to cut activation memory further.

Combined with **data parallelism** you get **3D parallelism**: `world_size = TP × PP × DP`. This is how GPT-3-, PaLM-, and Llama-scale models are actually trained. **Reach for Megatron** when a model is too large for one GPU and you control a multi-GPU/multi-node cluster. **Don't** reach for it for models that fit on one GPU (use DDP) or for inference (use vLLM/TensorRT-LLM — Megatron is a *training* stack). Today most people consume these ideas through **Megatron-Core** (the library) or wrappers like **NeMo**, **Megatron-DeepSpeed**, and HF **Nanotron**.

## 2. Mental Model

**Tensor parallelism = factor one big matmul into independent column/row pieces, then add the pieces back.** The transformer MLP `Z = (GELU(X·A))·B` shards into two matmuls glued by one all-reduce:

```
                 GPU 0                         GPU 1
            ┌───────────────┐            ┌───────────────┐
   X ──────►│ A₀ (cols 0..) │   X ──────►│ A₁ (..cols)   │   A split by COLUMN
            │   GELU         │            │   GELU         │  (no comm — GELU is
            │ Y₀ = GELU(XA₀) │            │ Y₁ = GELU(XA₁) │   element-wise)
            │ B₀ (rows 0..)  │            │ B₁ (..rows)   │   B split by ROW
            │ Z₀ = Y₀·B₀     │            │ Z₁ = Y₁·B₁     │
            └───────┬────────┘            └───────┬────────┘
                    └──────── all-reduce(+) ──────┘   →  Z = Z₀ + Z₁
```

The trick: make the **first** matmul **column-parallel** (no communication needed, because GELU acts element-wise on independent columns) and the **second** **row-parallel** (one all-reduce to sum the partial outputs). Attention shards the same way: QKV projection is column-parallel *by head*, the output projection is row-parallel. Result: **one all-reduce in the forward pass and one in the backward pass per block** — and the output is bit-for-bit what the un-sharded layer would produce.

Pipeline parallelism is the orthogonal axis: stack the layers into stages and stream micro-batches through them like an assembly line, overlapping stages to keep every GPU busy.

## 3. Key Concepts

- **Tensor (intra-layer) parallelism** — split individual weight matrices across GPUs. **Column-parallel** splits a `Linear` by output features (each GPU owns some output columns, no comm before an element-wise op); **row-parallel** splits by input features (each GPU produces a partial sum, glued by an **all-reduce**). MLP = column→row; attention = column-by-head→row.
- **Pipeline (inter-layer) parallelism** — split the layer stack into sequential **stages** on different devices. Activations and gradients pass only at stage boundaries (cheap), but the fill/drain of the pipeline wastes time — the **bubble**.
- **Micro-batching & schedules** — split each global batch into **micro-batches** so stages overlap. Bubble fraction ≈ `(p − 1) / (m + p − 1)` for `p` stages and `m` micro-batches → use `m ≫ p`. Megatron's **interleaved 1F1B** schedule shrinks the bubble further by giving each device several non-contiguous layer chunks.
- **Sequence parallelism** — shard the residual-path ops that TP leaves replicated (LayerNorm, dropout) along the **sequence** dimension, converting TP's all-reduce into a cheaper reduce-scatter + all-gather and cutting activation memory ~`1/t`.
- **3D parallelism** — `world_size = TP × PP × DP`. Rule of thumb: **TP inside a node** (needs NVLink bandwidth, keep ≤ 8), **PP across nodes**, **DP on top** for throughput. **ZeRO/distributed optimizer** then shards optimizer state across the DP group.
- **Activation recomputation (checkpointing)** — recompute activations in the backward pass instead of storing them, trading compute for memory. Essential at scale; often selective (only the attention block).
- **`global_batch = micro_batch × grad_accum × DP`** — the four knobs (TP, PP, DP, micro-batch) must satisfy memory *and* this batch-size identity simultaneously. Tuning them is the core of configuring a Megatron run.

## 4. Setup

Real Megatron-LM needs **multiple NVIDIA GPUs, NCCL, CUDA, and `apex`/Transformer Engine** — you launch it with `torchrun` across a cluster, not in a single notebook cell. That full stack can't run on CPU. But the *ideas* are just linear algebra, so this notebook **simulates** tensor parallelism on CPU with plain PyTorch: we shard weight matrices into "per-GPU" pieces in a Python list and reproduce Megatron's column/row-parallel math exactly, asserting it equals the dense layer.

```bash
# What a real install looks like (multi-GPU box, not run here):
pip install megatron-core          # the reusable library
# or clone the full training framework:
git clone https://github.com/NVIDIA/Megatron-LM
# launch (example): 8-way tensor parallel on one node
torchrun --nproc_per_node=8 pretrain_gpt.py --tensor-model-parallel-size 8 ...
```

The cell below just needs CPU PyTorch + NumPy.

In [1]:
import os
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
print("torch:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu (simulating GPUs in a list)")
# Real Megatron reads these from torchrun's env; we just print the shape of the idea.
print("TENSOR_PARALLEL_SIZE (env, optional):", os.getenv("TENSOR_PARALLEL_SIZE", "unset -> simulate"))

torch: 2.12.1
device: cpu (simulating GPUs in a list)
TENSOR_PARALLEL_SIZE (env, optional): unset -> simulate


## 5. Worked Examples

### Example 1 — Tensor-parallel MLP reproduces the dense MLP exactly

This is the heart of Megatron. We build a dense transformer MLP `Z = GELU(X·A)·B`, then shard it across `TP` simulated GPUs the Megatron way — **A column-parallel, B row-parallel** — and show the all-reduced result is numerically identical (to floating-point) to the dense layer. No approximation: tensor parallelism changes *where* the arithmetic happens, never *what* it computes.

In [2]:
B, S, h = 2, 4, 16          # batch, seq, hidden
ffn = 4 * h                  # MLP inner dim (the GPT 4x expansion)
TP = 4                       # pretend we have 4 GPUs

X = torch.randn(B, S, h)
A = torch.randn(h, ffn)      # up-projection   (h   -> 4h)
Bw = torch.randn(ffn, h)     # down-projection (4h  -> h)

# --- dense reference (what one big GPU would compute) ---
Z_dense = F.gelu(X @ A) @ Bw

# --- Megatron tensor-parallel version across TP shards ---
A_shards = torch.chunk(A, TP, dim=1)    # COLUMN-parallel: split A by output cols
B_shards = torch.chunk(Bw, TP, dim=0)   # ROW-parallel:    split B by input rows

partials = []
for gpu in range(TP):
    # Each GPU: local up-proj -> GELU (element-wise, no comm) -> local down-proj
    Y_i = F.gelu(X @ A_shards[gpu])     # (B, S, ffn/TP)  lives only on this GPU
    Z_i = Y_i @ B_shards[gpu]           # (B, S, h)       partial output
    partials.append(Z_i)

Z_tp = torch.stack(partials).sum(dim=0)  # <-- the ONE all-reduce(+) per block

print("max |dense - tensor_parallel| =", (Z_dense - Z_tp).abs().max().item())
print("identical (atol=1e-5):", torch.allclose(Z_dense, Z_tp, atol=1e-5))
print(f"per-GPU A shard {tuple(A_shards[0].shape)}, B shard {tuple(B_shards[0].shape)} "
      f"-> each GPU stores 1/{TP} of the weights")

max |dense - tensor_parallel| = 7.62939453125e-06
identical (atol=1e-5): True
per-GPU A shard (16, 16), B shard (16, 16) -> each GPU stores 1/4 of the weights


The two halves of the MLP need exactly **one communication** (the final sum = an `all_reduce`). That's why Megatron keeps TP inside one NVLink-connected node: this all-reduce happens on *every* block in *both* forward and backward passes, so it must be fast.

### Example 2 — Multi-head attention shards by head (column-parallel QKV)

Attention is even cleaner: split it **by attention head**. Each GPU owns a contiguous group of heads, runs the full attention for those heads independently, and the row-parallel output projection all-reduces the result. Below we verify a 4-head attention split across 2 GPUs (2 heads each) matches single-device attention.

In [3]:
B, S, h, n_head = 1, 5, 16, 4
d_head = h // n_head
TP = 2                        # 2 GPUs, 2 heads each

X = torch.randn(B, S, h)
Wq = torch.randn(h, h); Wk = torch.randn(h, h); Wv = torch.randn(h, h)
Wo = torch.randn(h, h)        # output projection (row-parallel)

def attention(x, wq, wk, wv, nh):
    B, S, _ = x.shape
    q, k, v = x @ wq, x @ wk, x @ wv
    dh = q.shape[-1] // nh          # head dim from the (possibly sharded) projection
    # reshape to heads: (B, nh, S, dh)
    q, k, v = (t.view(B, S, nh, dh).transpose(1, 2) for t in (q, k, v))
    att = (q @ k.transpose(-2, -1)) / math.sqrt(dh)
    out = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, S, nh * dh)
    return out

# --- dense reference ---
ctx_dense = attention(X, Wq, Wk, Wv, n_head)
Z_dense = ctx_dense @ Wo

# --- tensor-parallel: each GPU owns n_head/TP heads (column-parallel QKV) ---
heads_per_gpu = n_head // TP
cols = heads_per_gpu * d_head            # columns of Wq/Wk/Wv this GPU owns
partials = []
for gpu in range(TP):
    sl = slice(gpu * cols, (gpu + 1) * cols)
    ctx_i = attention(X, Wq[:, sl], Wk[:, sl], Wv[:, sl], heads_per_gpu)  # local heads
    Z_i = ctx_i @ Wo[sl, :]              # ROW-parallel output proj (partial)
    partials.append(Z_i)
Z_tp = torch.stack(partials).sum(0)      # all-reduce(+)

print("max |dense - tensor_parallel| =", (Z_dense - Z_tp).abs().max().item())
print("identical (atol=1e-5):", torch.allclose(Z_dense, Z_tp, atol=1e-5))
print(f"{n_head} heads / {TP} GPUs = {heads_per_gpu} heads per GPU")

max |dense - tensor_parallel| = 1.9073486328125e-06
identical (atol=1e-5): True
4 heads / 2 GPUs = 2 heads per GPU


### Example 3 — Sizing a 3D-parallel run: memory and the pipeline bubble

You don't *run* a 70B model on CPU, but the arithmetic that decides the parallel layout is just division. This cell does the back-of-the-envelope a Megatron user actually does: given a model and a GPU budget, pick `TP × PP × DP` and check (a) per-GPU parameter memory and (b) the pipeline bubble.

In [4]:
def transformer_params(n_layers, h, vocab, seq):
    # per layer: attention 4h^2 + MLP 8h^2 = 12 h^2 (ignoring small bias/LN terms)
    per_layer = 12 * h * h
    embeds = vocab * h + seq * h
    return n_layers * per_layer + embeds

# A Llama-2-13B-ish config
n_layers, h, vocab, seq = 40, 5120, 32000, 4096
P = transformer_params(n_layers, h, vocab, seq)
print(f"total params ~ {P/1e9:.1f}B")

# fp16 weights + fp16 grads + fp32 Adam (m, v, master) = ~16 bytes/param at full replication
bytes_per_param = 2 + 2 + 12
full_replica_gb = P * bytes_per_param / 1e9
print(f"memory if NOT sharded: ~{full_replica_gb:.0f} GB  (won't fit an 80GB GPU)\n")

for TP, PP, DP in [(8, 1, 1), (8, 2, 1), (4, 2, 2)]:
    world = TP * PP * DP
    # TP shards weights+grads+optimizer; PP shards by layer count; DP shards optimizer only (ZeRO-1)
    weight_gb = (P * 4 / 1e9) / (TP * PP)              # fp16 weight+grad, sharded by TP*PP
    optim_gb  = (P * 12 / 1e9) / (TP * PP * DP)        # fp32 Adam states, also sharded over DP
    per_gpu = weight_gb + optim_gb
    layers_per_stage = math.ceil(n_layers / PP)
    print(f"TP={TP} PP={PP} DP={DP}  world={world:2d} GPUs | "
          f"~{per_gpu:5.1f} GB/GPU | {layers_per_stage} layers/stage")

# Pipeline bubble: fraction of time stages sit idle filling/draining.
print("\nPipeline bubble vs #micro-batches (PP=4 stages):")
p = 4
for m in (1, 4, 8, 16, 64):
    bubble = (p - 1) / (m + p - 1)
    print(f"  m={m:2d} micro-batches -> {bubble*100:4.1f}% idle")

total params ~ 12.8B
memory if NOT sharded: ~204 GB  (won't fit an 80GB GPU)

TP=8 PP=1 DP=1  world= 8 GPUs | ~ 25.5 GB/GPU | 40 layers/stage
TP=8 PP=2 DP=1  world=16 GPUs | ~ 12.8 GB/GPU | 20 layers/stage
TP=4 PP=2 DP=2  world=16 GPUs | ~ 16.0 GB/GPU | 20 layers/stage

Pipeline bubble vs #micro-batches (PP=4 stages):
  m= 1 micro-batches -> 75.0% idle
  m= 4 micro-batches -> 42.9% idle
  m= 8 micro-batches -> 27.3% idle
  m=16 micro-batches -> 15.8% idle
  m=64 micro-batches ->  4.5% idle


Two takeaways the output makes concrete: **(1)** sharding turns an impossible ~200 GB footprint into something that fits per GPU — and optimizer state (the fp32 Adam moments) dominates, which is why the **distributed optimizer / ZeRO** matters as much as TP. **(2)** The pipeline bubble shrinks as you add micro-batches (`m ≫ p`), which is the whole reason Megatron pushes large global batches and the interleaved 1F1B schedule.

## 6. Gotchas & Pitfalls

- **TP all-reduce is bandwidth-bound — keep TP inside one node.** Every transformer block does an all-reduce in forward *and* backward. Across slow inter-node links this dominates step time. Rule: **TP ≤ #GPUs-per-node** (usually ≤ 8, the NVLink domain); use **PP across nodes**.
- **Pipeline bubbles waste GPUs.** With `m` micro-batches and `p` stages, a fraction `(p−1)/(m+p−1)` of every step is idle. Too few micro-batches (or too many stages) and you pay for GPUs doing nothing. Increase `m`, use interleaved 1F1B, or prefer TP/DP over deep PP.
- **The batch-size identity must hold:** `global_batch = micro_batch × grad_accum × DP`. Change TP/PP/DP and you silently change the effective batch (and learning dynamics) unless you re-balance grad accumulation. A very common "my loss curve changed" bug.
- **Vocab/embedding & LayerNorm need care.** The embedding and final logit projection are sharded over the **vocabulary** dimension (with a parallel cross-entropy), and TP leaves LayerNorm/dropout replicated — **sequence parallelism** exists precisely to shard those. Forgetting SP leaves a big chunk of activation memory un-sharded.
- **Don't confuse TP with ZeRO/FSDP.** ZeRO/FSDP shard *for memory* and gather full weights just-in-time for each layer's compute (it's data-parallel underneath). TP keeps weights permanently sharded and shards the *compute* too. They compose, but they are different axes — TP changes the math layout, ZeRO doesn't.
- **Activation memory, not weights, usually OOMs you.** At long sequence length activations dwarf weights. Reach for **activation recomputation** (selective is often enough) before adding more PP stages.
- **It's a training framework, not a server.** Megatron-LM produces checkpoints; serving them is a separate stack (TensorRT-LLM / vLLM). Don't try to do low-latency inference with the training pipeline.
- **RNG/determinism across shards.** Dropout and init must use *tensor-parallel-aware* seeding so each shard's randomness is consistent; naive `torch.manual_seed` per rank breaks reproducibility and correctness. Megatron handles this — your custom layers must too.

## 7. When to Use vs Alternatives

| Situation | Use | Why |
|---|---|---|
| Model **fits** on one GPU, want throughput | **DDP** (plain data parallel) | Simplest; no model sharding needed. |
| Model **barely** exceeds one GPU; minimal code change | **ZeRO-3 / PyTorch FSDP** | Shards weights+optimizer for memory, stays data-parallel — no model surgery. |
| Model **far** exceeds one GPU; you own a cluster | **Megatron-LM / Megatron-Core** | TP+PP+SP give the best compute efficiency (MFU) at extreme scale. |
| Want Megatron's parallelism + ZeRO ergonomics | **Megatron-DeepSpeed / Nanotron** | Combine TP/PP with DeepSpeed ZeRO or HF tooling. |
| Higher-level, config-driven training | **NVIDIA NeMo** | Wraps Megatron-Core with data, recipes, and launchers. |
| **Inference / serving** a trained model | **TensorRT-LLM, vLLM, SGLang** | Megatron is for training; these do paged-attention, batching, quantization. |

**Honest trade-offs.** Megatron extracts the highest hardware utilization at frontier scale, but it's *operationally heavy*: you reason about a 3D GPU grid, every layer must be parallelism-aware, and debugging spans nodes. **FSDP/ZeRO-3 is the right default** until pure data/optimizer sharding stops fitting or its all-gather overhead hurts — then you add **TP** (within a node) and finally **PP** (across nodes). Most people never write raw Megatron-LM; they get its ideas via Megatron-Core, NeMo, or FSDP. See also [`flash-attention`](./flash-attention.ipynb) (the kernel each shard runs), [`vllm`](./vllm.ipynb) (serving the result), and [`qlora`](./qlora.ipynb) (the fit-on-one-GPU alternative for *fine-tuning*).

## 8. Resources

- **Megatron-LM repo** — the framework and example training scripts: <https://github.com/NVIDIA/Megatron-LM>
- **Megatron-Core docs** — the reusable library most people actually import: <https://docs.nvidia.com/megatron-core/developer-guide/latest/>
- **"Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism"** (Shoeybi et al., 2019) — the tensor-parallel paper: <https://arxiv.org/abs/1909.08053>
- **"Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM"** (Narayanan et al., 2021) — 3D parallelism + interleaved pipeline schedule: <https://arxiv.org/abs/2104.04473>
- **"Reducing Activation Recomputation in Large Transformer Models"** (Korthikanti et al., 2022) — sequence parallelism + selective recompute: <https://arxiv.org/abs/2205.05198>
- **HF blog — "The Technology Behind BLOOM Training"** — a clear walkthrough of TP/PP/DP/ZeRO in practice: <https://huggingface.co/blog/bloom-megatron-deepspeed>